In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from tqdm import tqdm
import time
import re
from urllib.parse import quote

from selenium import webdriver
import csv
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from datetime import datetime
import re
import os
 
def preprocess_sentence_kr(w):
  w = w.strip()
  w = re.sub(r"[^0-9가-힣?.!,¿]+", " ", w)
  w = w.strip()
  return w



In [2]:
# 요리 전/중/후 폴더 구분하여 생성

folder_path_1 = "./results/요리 전" # 생성하려는 폴더 경로
os.makedirs(folder_path_1, exist_ok=True)

folder_path_2 = "./results/요리 중" # 생성하려는 폴더 경로
os.makedirs(folder_path_2, exist_ok=True)

folder_path_3 = "./results/요리 후" # 생성하려는 폴더 경로
os.makedirs(folder_path_3, exist_ok=True)

1. 특정 `네이버 카페` 들어가기
2. `전체글보기` 메뉴 선택
3. url 복사

In [ ]:
urls_dict = dict()

# urls_dict['레몬테라스'] = 'https://cafe.naver.com/f-e/cafes/10298136/menus/0?viewType=L'
# urls_dict['슬기로운 주부생활'] = 'https://cafe.naver.com/f-e/cafes/25774568/menus/0?viewType=L'
# urls_dict['구리남양주맘'] = 'https://cafe.naver.com/f-e/cafes/13815568/menus/0?viewType=L'
# urls_dict['퍼플맘의 스마트육아'] = 'https://cafe.naver.com/f-e/cafes/25114361/menus/0?viewType=L'
# urls_dict['대구맘 365'] = 'https://cafe.naver.com/f-e/cafes/24000254/menus/0?viewType=L'
# urls_dict['맘스홀릭 베이비'] = 'https://cafe.naver.com/f-e/cafes/10094499/menus/0?viewType=L'
urls_dict['세종맘 대전맘'] = 'https://cafe.naver.com/f-e/cafes/10797658/menus/0?viewType=L'

In [ ]:
# 요리 전
search_keywords_lst = ['냉장고 정리', '장보기', '저녁 메뉴', '손질', '유통기한', '요리 준비']

# 요리 중
search_keywords_lst_2 = ['요리 실패', '불 조절', '요리 냄새', '인덕션', '광파오븐', '굽기', '끓이기', '요리 환기']

# 요리 후
search_keywords_lst_3 = ['주방 정리', '식기세척기', '식세기', '음식물 처리', '잔반', '기름 때', '설거지']


### Macro (요리 전)

In [ ]:
for cafen, url0 in tqdm(urls_dict.items()):
    for search_keyword_ in tqdm(search_keywords_lst):
        print('='*20)
        print(f'[{cafen}] [{search_keyword_}] - Ongoing...')
        
        keyword = quote(search_keyword_)

        # region => 네이버 로그인
        url_login='https://nid.naver.com/nidlogin.login'
        id='wnsghd100'
        pw='Golem2418!!'

        chrome_options = Options()
        chrome_options.add_experimental_option("detach", True)
        chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
        # chrome_options.add_argument("headless") # headless option
        
        # 1. Chrome 옵션 설정 (리눅스에서는 headless 모드 추천)
        chrome_options.add_argument('--headless') # 브라우저 창을 띄우지 않고 실행
        chrome_options.add_argument('--no-sandbox') # 리눅스에서 권장되는 옵션
        chrome_options.add_argument('--disable-dev-shm-usage') # 공유 메모리 사용 비활성화
        
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url_login)

        driver.implicitly_wait(2)

        driver.execute_script(f"document.getElementsByName('id')[0].value=\'{id}\'")
        driver.execute_script(f"document.getElementsByName('pw')[0].value=\'{pw}\'")
        time.sleep(0.5)

        driver.find_element(by=By.XPATH,value='//*[@id="log.login"]').click()
        time.sleep(1)
        # endregion

        option = 999999 # page
        cnts = 0 # 게시글 개수

        # '요리 전' 폴더
        f = open(f'{folder_path_1}/네이버_카페_{cafen}_{search_keyword_}.txt', 'w', encoding='utf-8')

        # 데이터프레임 저장
        contents_lst = [] # 내용(글) 저장
        titles_lst = [] # 게시글 제목 저장
        dates_lst = [] # 게시글 날짜 저장

        for p in range(1, option+1):
            print('Page: ', p)
            # url = f'{url0}&ta=SUBJECT&q={keyword}&page={p}' # 제목, 기간 필터링 X
            url = f'{url0}&ta=ARTICLE_COMMENT&q={keyword}&page={p}' # 글+댓글, 기간 필터링 X
            # print(url)
            
            try:
                
                driver.get(url)
                time.sleep(1)
                
                contents = driver.find_elements(By.CSS_SELECTOR, 'a.article')
                contents_dates = driver.find_elements(By.CSS_SELECTOR, 'tbody > tr > td:nth-of-type(4).td_normal')
                
                href_list = [c.get_attribute('href') for c in contents]
                
                titles_lst0 = [c.text for c in contents]
                dates_lst0 = ['-'.join(d.text.split('.'))[:-1] for d in contents_dates]
            

                print(f"Page {p}, 게시글(url) 개수: ", len(href_list))

                # 게시글이 없으면(=마지막 페이지) 중단
                if len(href_list) == 0:
                    print('Last page finished')
                    break
                

                # for idx, href in enumerate(tqdm(href_list)):
                for idx, href in enumerate(href_list):
                
                    driver.get(href) # 해당 게시글 들어감
                    time.sleep(1)
                    
                    # iframe
                    driver.switch_to.frame('cafe_main')
                    time.sleep(0.5)
                    

                    # content_ = driver.find_elements(By.CSS_SELECTOR, 'div.se-main-container')
                    # content_ = driver.find_element(By.CSS_SELECTOR, 'div.article_container')
                    content_ = driver.find_elements(By.CSS_SELECTOR, 'div.article_viewer')
                    if content_ != []:
                        f.write(content_[0].text)
                        
                        # df
                        contents_lst.append(content_[0].text)
                        titles_lst.append(titles_lst0[idx])
                        dates_lst.append(dates_lst0[idx])
                        
                        cnts+= 1
                        
                    else: # 높은 등급멤버(or 회원)만 읽을 수 있는 글 일때 넘어감
                        print('(SKIP) 회원 전용 글')
                        
                        
                    driver.switch_to.default_content()
                    time.sleep(0.5)
                    # driver.back()
                    # time.sleep(0.5)
                
                    
            except:
                print('오류 발생 (중간에 중단)')
                break

        f.close()
        driver.quit()

        print(f"[{cafen}] [{search_keyword_}] 크롤링 완료 게시글 개수: ", cnts)
        
        
        # dataframe 저장
        df = pd.DataFrame([titles_lst, dates_lst, contents_lst]).T
        df.columns = ['Title', 'Date', 'Contents']
        df = df.dropna() # 결측치 포함 행 제외
        df.to_excel(f'{folder_path_1}/네이버_카페_{cafen}_{search_keyword_}.xlsx', index=False)
        
        print('='*20)

  0%|          | 0/2 [00:00<?, ?it/s]

[레몬테라스] [냉장고 정리] - Ongoing...
Page:  1
Page 1, 게시글(url) 개수:  15
Page:  2
Page 2, 게시글(url) 개수:  15
Page:  3
Page 3, 게시글(url) 개수:  15
Page:  4
Page 4, 게시글(url) 개수:  15
Page:  5
Page 5, 게시글(url) 개수:  15
Page:  6
Page 6, 게시글(url) 개수:  15
Page:  7
Page 7, 게시글(url) 개수:  15
Page:  8
Page 8, 게시글(url) 개수:  15
Page:  9
Page 9, 게시글(url) 개수:  15
Page:  10
Page 10, 게시글(url) 개수:  15
Page:  11
Page 11, 게시글(url) 개수:  15
Page:  12
Page 12, 게시글(url) 개수:  15
Page:  13
Page 13, 게시글(url) 개수:  15
Page:  14
Page 14, 게시글(url) 개수:  15
Page:  15
Page 15, 게시글(url) 개수:  15
Page:  16
Page 16, 게시글(url) 개수:  15
Page:  17
Page 17, 게시글(url) 개수:  15
Page:  18
Page 18, 게시글(url) 개수:  15
Page:  19
Page 19, 게시글(url) 개수:  15
Page:  20
Page 20, 게시글(url) 개수:  15
Page:  21
Page 21, 게시글(url) 개수:  15
Page:  22
Page 22, 게시글(url) 개수:  15
Page:  23
Page 23, 게시글(url) 개수:  15
Page:  24
Page 24, 게시글(url) 개수:  15
Page:  25
Page 25, 게시글(url) 개수:  15
Page:  26
Page 26, 게시글(url) 개수:  15
Page:  27
Page 27, 게시글(url) 개수:  15
Page:  28
Page 2

### Macro (요리 중)

In [ ]:
for cafen, url0 in tqdm(urls_dict.items()):
    for search_keyword_ in tqdm(search_keywords_lst_2):
        print('='*20)
        print(f'[{cafen}] [{search_keyword_}] - Ongoing...')
        
        keyword = quote(search_keyword_)

        # region => 네이버 로그인
        url_login='https://nid.naver.com/nidlogin.login'
        id='wnsghd100'
        pw='Golem2418!!'

        chrome_options = Options()
        chrome_options.add_experimental_option("detach", True)
        chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
        # chrome_options.add_argument("headless") # headless option
        
        # 1. Chrome 옵션 설정 (리눅스에서는 headless 모드 추천)
        chrome_options.add_argument('--headless') # 브라우저 창을 띄우지 않고 실행
        chrome_options.add_argument('--no-sandbox') # 리눅스에서 권장되는 옵션
        chrome_options.add_argument('--disable-dev-shm-usage') # 공유 메모리 사용 비활성화
        
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url_login)

        driver.implicitly_wait(2)

        driver.execute_script(f"document.getElementsByName('id')[0].value=\'{id}\'")
        driver.execute_script(f"document.getElementsByName('pw')[0].value=\'{pw}\'")
        time.sleep(0.5)

        driver.find_element(by=By.XPATH,value='//*[@id="log.login"]').click()
        time.sleep(1)
        # endregion

        option = 999999 # page
        cnts = 0 # 게시글 개수

        # '요리 전' 폴더
        f = open(f'{folder_path_2}/네이버_카페_{cafen}_{search_keyword_}.txt', 'w', encoding='utf-8')

        # 데이터프레임 저장
        contents_lst = [] # 내용(글) 저장
        titles_lst = [] # 게시글 제목 저장
        dates_lst = [] # 게시글 날짜 저장

        for p in range(1, option+1):
            print('Page: ', p)
            # url = f'{url0}&ta=SUBJECT&q={keyword}&page={p}' # 제목, 기간 필터링 X
            url = f'{url0}&ta=ARTICLE_COMMENT&q={keyword}&page={p}' # 글+댓글, 기간 필터링 X
            # print(url)
            
            try:
                
                driver.get(url)
                time.sleep(1)
                
                contents = driver.find_elements(By.CSS_SELECTOR, 'a.article')
                contents_dates = driver.find_elements(By.CSS_SELECTOR, 'tbody > tr > td:nth-of-type(4).td_normal')
                
                href_list = [c.get_attribute('href') for c in contents]
                
                titles_lst0 = [c.text for c in contents]
                dates_lst0 = ['-'.join(d.text.split('.'))[:-1] for d in contents_dates]
            

                print(f"Page {p}, 게시글(url) 개수: ", len(href_list))

                # 게시글이 없으면(=마지막 페이지) 중단
                if len(href_list) == 0:
                    print('Last page finished')
                    break
                

                # for idx, href in enumerate(tqdm(href_list)):
                for idx, href in enumerate(href_list):
                
                    driver.get(href) # 해당 게시글 들어감
                    time.sleep(1)
                    
                    # iframe
                    driver.switch_to.frame('cafe_main')
                    time.sleep(0.5)
                    

                    # content_ = driver.find_elements(By.CSS_SELECTOR, 'div.se-main-container')
                    # content_ = driver.find_element(By.CSS_SELECTOR, 'div.article_container')
                    content_ = driver.find_elements(By.CSS_SELECTOR, 'div.article_viewer')
                    if content_ != []:
                        f.write(content_[0].text)
                        
                        # df
                        contents_lst.append(content_[0].text)
                        titles_lst.append(titles_lst0[idx])
                        dates_lst.append(dates_lst0[idx])
                        
                        cnts+= 1
                        
                    else: # 높은 등급멤버(or 회원)만 읽을 수 있는 글 일때 넘어감
                        print('(SKIP) 회원 전용 글')
                        
                        
                    driver.switch_to.default_content()
                    time.sleep(0.5)
                    # driver.back()
                    # time.sleep(0.5)
                
                    
            except:
                print('오류 발생 (중간에 중단)')
                break

        f.close()
        driver.quit()

        print(f"[{cafen}] [{search_keyword_}] 크롤링 완료 게시글 개수: ", cnts)
        
        
        # dataframe 저장
        df = pd.DataFrame([titles_lst, dates_lst, contents_lst]).T
        df.columns = ['Title', 'Date', 'Contents']
        df = df.dropna() # 결측치 포함 행 제외
        df.to_excel(f'{folder_path_2}/네이버_카페_{cafen}_{search_keyword_}.xlsx', index=False)
        
        print('='*20)

### Macro (요리 후)

In [ ]:
for cafen, url0 in tqdm(urls_dict.items()):
    for search_keyword_ in tqdm(search_keywords_lst_3):
        print('='*20)
        print(f'[{cafen}] [{search_keyword_}] - Ongoing...')
        
        keyword = quote(search_keyword_)

        # region => 네이버 로그인
        url_login='https://nid.naver.com/nidlogin.login'
        id='wnsghd100'
        pw='Golem2418!!'

        chrome_options = Options()
        chrome_options.add_experimental_option("detach", True)
        chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
        # chrome_options.add_argument("headless") # headless option
        
        # 1. Chrome 옵션 설정 (리눅스에서는 headless 모드 추천)
        chrome_options.add_argument('--headless') # 브라우저 창을 띄우지 않고 실행
        chrome_options.add_argument('--no-sandbox') # 리눅스에서 권장되는 옵션
        chrome_options.add_argument('--disable-dev-shm-usage') # 공유 메모리 사용 비활성화
        
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url_login)

        driver.implicitly_wait(2)

        driver.execute_script(f"document.getElementsByName('id')[0].value=\'{id}\'")
        driver.execute_script(f"document.getElementsByName('pw')[0].value=\'{pw}\'")
        time.sleep(0.5)

        driver.find_element(by=By.XPATH,value='//*[@id="log.login"]').click()
        time.sleep(1)
        # endregion

        option = 999999 # page
        cnts = 0 # 게시글 개수

        # '요리 전' 폴더
        f = open(f'{folder_path_3}/네이버_카페_{cafen}_{search_keyword_}.txt', 'w', encoding='utf-8')

        # 데이터프레임 저장
        contents_lst = [] # 내용(글) 저장
        titles_lst = [] # 게시글 제목 저장
        dates_lst = [] # 게시글 날짜 저장

        for p in range(1, option+1):
            print('Page: ', p)
            # url = f'{url0}&ta=SUBJECT&q={keyword}&page={p}' # 제목, 기간 필터링 X
            url = f'{url0}&ta=ARTICLE_COMMENT&q={keyword}&page={p}' # 글+댓글, 기간 필터링 X
            # print(url)
            
            try:
                
                driver.get(url)
                time.sleep(1)
                
                contents = driver.find_elements(By.CSS_SELECTOR, 'a.article')
                contents_dates = driver.find_elements(By.CSS_SELECTOR, 'tbody > tr > td:nth-of-type(4).td_normal')
                
                href_list = [c.get_attribute('href') for c in contents]
                
                titles_lst0 = [c.text for c in contents]
                dates_lst0 = ['-'.join(d.text.split('.'))[:-1] for d in contents_dates]
            

                print(f"Page {p}, 게시글(url) 개수: ", len(href_list))

                # 게시글이 없으면(=마지막 페이지) 중단
                if len(href_list) == 0:
                    print('Last page finished')
                    break
                

                # for idx, href in enumerate(tqdm(href_list)):
                for idx, href in enumerate(href_list):
                
                    driver.get(href) # 해당 게시글 들어감
                    time.sleep(1)
                    
                    # iframe
                    driver.switch_to.frame('cafe_main')
                    time.sleep(0.5)
                    

                    # content_ = driver.find_elements(By.CSS_SELECTOR, 'div.se-main-container')
                    # content_ = driver.find_element(By.CSS_SELECTOR, 'div.article_container')
                    content_ = driver.find_elements(By.CSS_SELECTOR, 'div.article_viewer')
                    if content_ != []:
                        f.write(content_[0].text)
                        
                        # df
                        contents_lst.append(content_[0].text)
                        titles_lst.append(titles_lst0[idx])
                        dates_lst.append(dates_lst0[idx])
                        
                        cnts+= 1
                        
                    else: # 높은 등급멤버(or 회원)만 읽을 수 있는 글 일때 넘어감
                        print('(SKIP) 회원 전용 글')
                        
                        
                    driver.switch_to.default_content()
                    time.sleep(0.5)
                    # driver.back()
                    # time.sleep(0.5)
                
                    
            except:
                print('오류 발생 (중간에 중단)')
                break

        f.close()
        driver.quit()

        print(f"[{cafen}] [{search_keyword_}] 크롤링 완료 게시글 개수: ", cnts)
        
        
        # dataframe 저장
        df = pd.DataFrame([titles_lst, dates_lst, contents_lst]).T
        df.columns = ['Title', 'Date', 'Contents']
        df = df.dropna() # 결측치 포함 행 제외
        df.to_excel(f'{folder_path_3}/네이버_카페_{cafen}_{search_keyword_}.xlsx', index=False)
        
        print('='*20)